In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
print(path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
path = os.path.join(path, 'Q3_data.csv')

df = pd.read_csv(path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
display(df.info())

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")
check_missing_values(df)

# Fill missing values with mean for each column

# Fill missing values with mean for each column
df = df.fillna(df.mean())

check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
# Task 3: Write your code here:
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")
check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
categorical_cols = df.select_dtypes(include=["object"]).columns

from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 5: Write your code here:

def check_imbalance(df, target):
    print("Target Dist")
    print(df[target].value_counts())

    plt.hist(x=df[target], bins=30)
    plt.show()



# Distribution
check_imbalance(df,"Target" )


In [ ]:
# Task 1: Write your code here:
X = df.drop("Target",axis=1)
y = df['Target']

In [ ]:
%pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import (
accuracy_score,
precision_score,
recall_score,
f1_score,
confusion_matrix,
classification_report
)
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold



# Define classification models
models = {
    "CatBoost Classifier": CatBoostClassifier(verbose=0,
learning_rate=0.0001,
random_state=42,
)
}


for model_name, model in models.items():
    scores_accuracy = []
    scores_precision = []
    scores_recall = []
    scores_f1 = []

    # Stratified 5-Fold Cross-Validation
    skf = StratifiedKFold(n_splits=5, shuffle=True)
    for train_index, test_index in skf.split(X, y):
        # Split data into training and testing sets
        X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
        y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
        # Train the model
        model.fit(X_Train, y_Train)
        # Predict on the test set
        y_pred = model.predict(X_Test)

        # Calculate metrics
        scores_f1.append(f1_score(y_Test, y_pred, average='weighted'))
        scores_accuracy.append(accuracy_score(y_Test, y_pred))

    # Print the results
    print(f"{model_name} F1-Score: {np.mean(scores_f1):.4f}")
        # accuracy = (predictions == y_test).sum().item() / y_test.size(0)
    print(f"{model_name} Accuracy: {np.mean(scores_accuracy):.4f}")
    print("\n")

# Retrieve CatBoost feature importances and sort them
catboost_model = models["CatBoost Classifier"]
catboost_importance = list(zip(X.columns, catboost_model.feature_importances_))
sorted_catboost_importance = sorted(catboost_importance, key=lambda x: x[1], reverse=True)



In [ ]:
# Task 1: Write your code here:
# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)

# Plot feature importances
plt.figure(figsize=(18, 18))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('CatBoost Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()


In [ ]:
# Task 2: Write your code here:
print(sorted_catboost_importance[0])

In [ ]:
X = X['P_2']



In [ ]:
y

In [ ]:
# Task Bonus: Write your code here:

# Define classification models
models = {
    "CatBoost Classifier": CatBoostClassifier(verbose=0
)
}


for model_name, model in models.items():
    scores_accuracy = []
    scores_f1 = []

    # Stratified 5-Fold Cross-Validation
    skf = StratifiedKFold(n_splits=5, shuffle=True)
    for train_index, test_index in skf.split(X, y):
        # Split data into training and testing sets
        X_Train, X_Test = X.iloc[train_index], X.iloc[test_index]
        y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
        # Train the model
        model.fit(X_Train, y_Train)
        # Predict on the test set
        y_pred = model.predict(X_Test)

        # Calculate metrics
        scores_f1.append(f1_score(y_Test, y_pred, average='weighted'))
        scores_accuracy.append(accuracy_score(y_Test, y_pred))

    # Print the results
    print(f"{model_name} F1-Score: {np.mean(scores_f1):.4f}")
    print(f"{model_name} Accuracy: {np.mean(scores_accuracy):.4f}")
    print("\n")


